# 탄자니아 급수시설 상태 예측 — EDA와 모델 학습 (Pump it Up)

- 데이터: 탄자니아 급수 지점의 설비·관리·위치 (59,400행 · 41열)
- 목표: 작동 상태 예측 (정상/수리필요/고장) — 3범주 분류
- 흐름: 불러오기 → 학습 전 확인 → EDA → 학습 → 해석
- 참고: 데이터 소개 data_PumpItUp.txt

- 이 데이터의 특징: **위장 결측** · 고차원 범주(수만 종)
  → AutoGluon 자동 인코딩의 위력과, 사람이 잡을 위장결측을 함께

## 1. 불러오기 — X와 y 병합

- 예측 변수(X)와 정답(y)이 **별도 파일**
- id 기준으로 병합해야 함
- ※ y_train_raw.csv 는 DrivenData에서 함께 받아야 함
  (X_train_raw.csv 만으로는 정답이 없어 학습 불가)

In [ ]:
import pandas as pd
import numpy as np

X = pd.read_csv("training_Set_values.csv")
y = pd.read_csv("training_Set_labels.csv")     # 정답 파일 (별도)
df = X.merge(y, on="id")
print(df.shape)          # (59400, 41)
df.head()

## 2. 학습 전 확인

- 이 데이터는 위장결측·식별자·고차원 범주를 사람이 처리

### 2-1. 숫자로 위장한 결측 (가장 중요)

- 여러 컬럼에서 0이 실제 값이 아니라 '미측정'
  - amount_tsh 70% · population 36% · gps_height 34%
  - construction_year 35%(연도 미상) · longitude 3%(범위 밖)
- AutoGluon은 0을 실제 값으로 학습 → 사람이 NaN으로 바꿔야
- **주의: latitude는 0이 없음 → 건드리지 말 것**

In [ ]:
# 0 → NaN (단, latitude 제외)
for c in ["construction_year", "gps_height",
          "longitude", "population"]:
    df[c] = df[c].replace(0, np.nan)

print("위장결측 처리 후 결측률:")
print(df[["construction_year","gps_height","longitude","population"]]
      .isna().mean().round(3))

### 2-2. 식별자 제거

- id는 예측에 무의미 → 제거
- recorded_by 등 상수 열은 AutoGluon이 자동 제거

In [ ]:
df = df.drop(columns=["id"])
print("id 제거 후:", df.shape)

### 2-3. 고차원 범주 (판단 필요)

- 일부 범주형 컬럼은 고유값이 수천~수만 종
  - wpt_name 37,399 · subvillage 19,287 · funder 1,896 ...
- 너무 많으면 학습이 무겁고 과적합 위험
- 제거하거나 상위 N개만 남기고 '기타'로 묶기

In [ ]:
# 고유값 개수 확인
high_card = {}
for c in df.select_dtypes(include="object").columns:
    n = df[c].nunique()
    if n > 1000:
        high_card[c] = n
print("고유값 1000종 이상:", high_card)

# 예: wpt_name 처럼 극단적인 것은 제거 (실습에선 판단)
df = df.drop(columns=["wpt_name", "subvillage"])
print("\n제거 후:", df.shape)

## 3. EDA

- 대규모(6만 행)라 표본으로 리포트

In [ ]:
from data_profiling import ProfileReport

sample = df.sample(10000, random_state=42)
profile = ProfileReport(sample, minimal=True, progress_bar=False)
profile.to_file("pump_eda.html")   # 브라우저에서 열기

### 3-1. 타깃 분포

In [ ]:
import matplotlib.pyplot as plt
df["status_group"].value_counts().plot(
    kind="bar", figsize=(6,3), title="pump status")
plt.show()
# 3범주 비율이 불균형 → 정확도만 보면 안 됨

### 3-2. 수량(quantity)과 상태

- 물이 마른(dry) 지점은 고장이 많은가

In [ ]:
pd.crosstab(df["quantity"], df["status_group"],
            normalize="index").round(2)

## 4. 학습

- 범주형 30개를 AutoGluon이 자동 인코딩 (사람이 안 함)
- 3범주 불균형 → 평가 지표 고려

In [ ]:
from autogluon.tabular import TabularPredictor
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(
    df, test_size=0.2, random_state=42,
    stratify=df["status_group"])

predictor = TabularPredictor(
    label="status_group",
    eval_metric="accuracy",   # balanced_accuracy 등도 비교
).fit(train_data, presets="medium_quality", time_limit=600)

In [ ]:
predictor.leaderboard(test_data)

## 5. 해석

In [ ]:
print(predictor.evaluate(test_data))

### 5-1. 변수 중요도

- 무엇이 급수시설 상태를 예측하는가
- quantity(수량)·연식·설비 유형이 상위에 오는가

In [ ]:
predictor.feature_importance(test_data).head(15)

## 정리

- X·y 별도 파일 → id로 병합
- 위장결측(0=미측정) → NaN 변환 (단 latitude 제외)
- 식별자 제거 · 고차원 범주 정리
- 범주형 30개는 AutoGluon이 자동 인코딩
- 학습·앙상블도 자동

- 이 데이터가 보여주는 것:
  자동 인코딩은 도구가, 위장결측 판단은 사람이
  → "핵심은 사람" (0의 의미를 아는 것은 사람 몫)